In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()

# Find the repository directory containing insulators/__init__.py
while not (repo_root / "insulators" / "__init__.py").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Could not find the repository root")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from insulators.catenary import catenary, catenary_length
from insulators.line import line_between_points
from insulators.sag import catenary_sag_vertical
from insulators.plotting import catenary_plot_full
from insulators.th_solver import Th_for_target_sag
from insulators.attachment_solver import get_attachment_points_for_Th
from insulators.span_solver import solve_span_for_target_sag
from insulators.plotting import catenary_plot_with_insulators
from insulators.equation_9 import equation_9_sag, equation_9_tension
from scripts.tensions import (
    Taxial_A,
    Taxial_B,
    Tv_A,
    Tv_B,
    conductor_length as span_conductor_length,
    distance_lowest_point_r,
)


In [ ]:

# Input Constants 
# note: kg = kg-force (essentially)

w = 34.91           #N/m
G = 160 * 9.81      #N     
L = 2               #m
E = 110000          #N/mm2              #6.18e9          #kg/m2 
diatomi =400        #mm2                #5.27            #cm2

# Suspension points
A = (0, 100) 
B = (62, 100 + 1.25)      #(593.70, 100 + 131.65)    #(144.54, 100-6.64) #

# target horizontal tension/ target sag
Th = 7526.26            #N
target_Th = Th          #N
target_sag = 3.87       #m  
target_fe = 3.26        #m

In [ ]:

############### usage ######################

print("#########################################")

result = solve_span_for_target_sag(
    A=A,
    B=B,
    w=w,
    target_sag= target_sag,
    L_left=L,
    L_right=L,
    G_left=G,
    G_right=G,
)

Th = result["Th"]
C = result["C"]
D = result["D"]


print(f'Th = {result["Th"]:.2f} N')
print(f'C = ({result["C"][0]:.2f}, {result["C"][1]:.2f})')
print(f'D = ({result["D"][0]:.2f}, {result["D"][1]:.2f})')
print("theta_left =", result["theta_left"].round(2), "rad")
print("theta_right =", result["theta_right"].round(2), "rad")
print("sag =", result["sag"].round(2), "m")
print("outer converged =", result["converged"])
print("inner converged =", result["attachment_converged"])

print("#########################################")

#cat = catenary(A, B, w, Th)
cat = result["catenary"]
lineAB = line_between_points(A, B)
lineCD = line_between_points(C, D)

sag_info = catenary_sag_vertical(cat, lineAB)

print("maximum sag =", sag_info["sag_max"].round(2), "m")
print("x at maximum sag =", sag_info["x_sag"].round(3), "m")
print("catenary y there =", sag_info["y_curve"].round(2), "m")
print(f"fe = {100 - sag_info["y_curve"]:.2} m")
print("line y there =", sag_info["y_line"].round(2), "m")

print("#########################################")

# Characteristic values for the idealized A-B and actual C-D catenaries
# scripts.tensions uses local span coordinates and returns downward vertical
# reactions as negative values. abs(...) gives the positive magnitudes used
# in the study table.
def span_characteristics(left, right, horizontal_tension):
    span = float(right[0] - left[0])
    height_difference = float(right[1] - left[1])
    horizontal_tension = float(horizontal_tension)

    return {
        "length": float(
            span_conductor_length(
                span,
                horizontal_tension,
                w,
                height_difference,
            )
        ),
        "Ty_left": abs(float(Tv_A(span, horizontal_tension, w, height_difference))),
        "Ty_right": abs(float(Tv_B(span, horizontal_tension, w, height_difference))),
        "S_left": float(Taxial_A(span, horizontal_tension, w, height_difference)),
        "S_right": float(Taxial_B(span, horizontal_tension, w, height_difference)),
        # Despite its historical name, this helper returns the vertex
        # coordinate measured from the left endpoint of the span.
        "x_vertex_from_left": float(
            distance_lowest_point_r(
                span,
                height_difference,
                horizontal_tension,
                w,
            )
        ),
    }


# Match the idealized A-B catenary to the same A-B sag as the solved actual span.
# This is the same convention used by catenary_plot_with_insulators().
Th_idealized = Th_for_target_sag(A, B, w, result["sag"])
idealized_values = span_characteristics(A, B, Th_idealized)
actual_values = span_characteristics(C, D, result["Th"])

conductor_length_idealized = idealized_values["length"]
conductor_length_actual = actual_values["length"]

Tya = idealized_values["Ty_left"]
Tyb = idealized_values["Ty_right"]
Tyc = actual_values["Ty_left"]
Tyd = actual_values["Ty_right"]

Sa = idealized_values["S_left"]
Sb = idealized_values["S_right"]
Sc = actual_values["S_left"]
Sd = actual_values["S_right"]

Xa = idealized_values["x_vertex_from_left"]
Xc = actual_values["x_vertex_from_left"]

print(f"Idealized horizontal tension = {Th_idealized:.2f} N")
print(f"Conductor length (idealized A-B) = {conductor_length_idealized:.2f} m")
print(f"Conductor length (actual C-D) = {conductor_length_actual:.2f} m")
print(f"Tya = {Tya:.2f} N")
print(f"Tyb = {Tyb:.2f} N")
print(f"Tyc = {Tyc:.2f} N")
print(f"Tyd = {Tyd:.2f} N")
print(f"Sa = {Sa:.2f} N")
print(f"Sb = {Sb:.2f} N")
print(f"Sc = {Sc:.2f} N")
print(f"Sd = {Sd:.2f} N")
print(f"Xa (distance A to idealized vertex) = {Xa:.2f} m")
print(f"Xc (distance C to actual vertex) = {Xc:.2f} m")
print(f"Paper-style signed x_A = {-Xa:.2f} m")

print("#########################################")

# Actual horizontal conductor span between attachments C and D
#a0 = float(result["D"][0] - result["C"][0])
a0 = float(B[0] - A[0])

sag_eq9 = equation_9_sag(
    a0=a0,
    w=w,
    L_ins=L,
    G_ins=G,
    tension=target_Th,
)

tension_eq9 = equation_9_tension(
    a0=a0,
    w=w,
    L_ins=L,
    G_ins=G,
    target_sag=target_sag,
)

tension_eq9_fe = equation_9_tension(
    a0=a0,
    w=w,
    L_ins=L,
    G_ins=G,
    target_sag=target_fe,
)

print(f"Equation 9 sag from tension ({target_Th} N) = {sag_eq9:.2f} m")
print(f"Equation 9 tension from target sag ({target_sag} m) = {tension_eq9:.2f} N")
print(f"Equation 9 tension from target fe ({target_fe} m) = {tension_eq9_fe:.2f} N")


print("#########################################")

#catenary_plot_full(cat, lineAB)

catenary_plot_with_insulators(
    cat=result["catenary"],
    A=A,
    B=B,
    C=result["C"],
    D=result["D"],
    show_actual=True,
    show_supports=False,
    show_attachments=True,
    show_chord=True,
    show_low_point=False,
    show_sag=True,
    show_idealized=True,
    show_idealized_sag=False,
    annotate_tensions=True,
    only_idealized=False,
    w=w,
    title="Αλυσοειδής",
    #figsize=(18, 11),
)



In [ ]:
from insulators.dxf_export import export_span_to_dxf

dxf_path = export_span_to_dxf(
    result=result,
    A=A,
    B=B,
    output_path="insulator_span.dxf",
    show_supports=True,
    show_attachments=True,
    show_chord=True,
    show_sag=True,
    show_idealized=True,
    show_idealized_sag=False,
    w=w,
    coordinate_precision=2,
)

print(dxf_path.resolve())